# PIPELINE 1 — Vietnamese side · Minh Mệnh Chính Yếu
Split PDF → render trang Hán → **VI OCR (Surya)** → spell-fix + clean → đẩy `output_p1/<vol>.zip`.
Chạy **cả 6 tập** trong 1 lần. Output là đầu vào cho **PIPELINE 2 (Hán)**.

Thứ tự: deps → **Restart** → verify → mount → config → loop 6 vol → push.


## 1. Deps (Surya stack — KHÔNG paddle) → rồi **Runtime ▸ Restart**


In [ ]:
# Surya 0.14.5 in-process. transformers PHẢI 4.51.3 (4.57 xóa QuantizedCacheConfig -> surya gãy).
# 1 lần pip, KHÔNG tách, KHÔNG Ctrl-C giữa chừng.
!pip -q install PyMuPDF opencv-python-headless pandas openpyxl tqdm
!pip -q install underthesea
!pip install "surya-ocr==0.14.5" "transformers==4.51.3" "tokenizers==0.21.1" "pillow>=10.2.0,<11.0.0"
import importlib.metadata as _m
_tv=_m.version("transformers"); print("transformers", _tv)
assert _tv=="4.51.3", f"transformers={_tv} != 4.51.3 (re-run cell, ĐỪNG Ctrl-C)"
print(">>> OK -> Runtime ▸ Restart runtime, rồi chạy từ cell VERIFY.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 93.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 78.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.9/176.9 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 108.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 127.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 119.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 126.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.5/226.5 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.4/99.4 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6

transformers 4.51.3
>>> OK -> Runtime ▸ Restart runtime, rồi chạy từ cell VERIFY.


### Verify GPU + Surya (sau restart)


In [ ]:
import torch, importlib.metadata as m
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| surya-ocr:", m.version("surya-ocr"))
from surya.recognition import RecognitionPredictor
from surya.detection import DetectionPredictor
det = DetectionPredictor(); rec = RecognitionPredictor()
print("Surya 0.14 API OK")


torch 2.11.0+cu128 | CUDA: True | surya-ocr: 0.14.5
Surya 0.14 API OK


## 2. Mount Drive + project (work root = LOCAL SSD, FUSE-safe)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
PROJECT = '/content/drive/MyDrive/minh_menh'   # code + data/<vol>.pdf + assets
MMCY    = '/content/mmcy'
os.environ['MMCY_ROOT'] = MMCY
%cd $PROJECT
!mkdir -p {MMCY}/data {MMCY}/out
!rsync -a {PROJECT}/assets/ {MMCY}/assets/
print('code:', PROJECT, '| work root (LOCAL):', MMCY)


Mounted at /content/drive
/content/drive/MyDrive/minh_menh
code: /content/drive/MyDrive/minh_menh | work root (LOCAL): /content/mmcy


## 3. Config — danh sách tập (VI OCR = Surya, không còn cờ recognizer)

In [ ]:
VOLS = ["vol1","vol2","vol3","vol4","vol5","vol6"]   # chạy hết. Bớt để test.
from pipeline import config
v = config.VIETNAMESE   # VI OCR = Surya only (vietocr/embedded đã bỏ)
print("VOLS:", VOLS)
# 3 pass chuẩn hóa (corrections.csv -> confusables -> spellcheck) LUÔN chạy ở step2.
print("spell_max_edit:", v["spell_max_edit_distance"], "| min_token_len:", v["spell_min_token_len"],
      "| crop_header:", v["crop_header"])
import subprocess, shutil, statistics as st
from pipeline.common import read_jsonl
def sh(args):
    print("  $", " ".join(args)); subprocess.run(args, cwd=PROJECT, check=True)

## 4. LOOP 6 tập — split + VI OCR + spell + clean


In [ ]:
import time
report = {}
for VOL in VOLS:
    t0 = time.time(); print(f"\n========== {VOL} ==========")
    # step1: split PDF + render trang Hán (LOCAL out/, không ghi Drive)
    shutil.copy2(f"{PROJECT}/data/{VOL}.pdf", f"{MMCY}/data/{VOL}.pdf")
    sh(["python","-m","pipeline.step1_split_pdf","--vol",VOL])
    # step2: VI OCR (Surya) + spell-fix + tách câu + vi_review. Xóa output cũ để lỗi LỘ ra.
    stale = config.OUT_DIR / VOL / "vi_sentences.jsonl"
    if stale.exists(): stale.unlink()
    sh(["python","-m","pipeline.step2_vietnamese","--vol",VOL])
    # clean: lọc rác OCR ở tầng câu VI
    sh(["python","-m","pipeline.clean_vi","--vol",VOL])
    # QA
    rows = list(read_jsonl(config.OUT_DIR / VOL / "vi_sentences.jsonl"))
    oov  = [r["oov_rate"] for r in rows if r.get("n_tokens")]
    bold = sum("<b>" in r["text"] for r in rows)
    assert rows and oov, f"{VOL}: step2 ra rỗng — xem log surya (chars>0?)."
    assert bold == 0, f"{VOL}: <b> tag -> đang đọc file embedded CŨ, không phải surya (check MMCY_ROOT)."
    report[VOL] = dict(n=len(rows), oov_mean=round(st.mean(oov),3), oov_hi=sum(o>0.3 for o in oov))
    print(f"  {VOL}: {len(rows)} câu | OOV mean {st.mean(oov):.3f} | OOV>0.3 {report[VOL]['oov_hi']} | {time.time()-t0:.0f}s")
print("\n== QA tổng ==")
for k,r in report.items(): print(f"  {k}: {r}")


## 5. Push — mỗi tập 1 zip → `output_p1/<vol>.zip` (1 lần ghi Drive/tập, FUSE-safe)


In [ ]:
# Bundle những gì PIPELINE 2 cần: pages_han (crop Hán cho Qwen) + split_manifest +
# vi_sentences (align) + vi_boxes (QA overlay) + pages_vi (QA) + vi_review/oov_vocab
# (hàng đợi soát VI — sinh ngay ở P1 để review tay trước alignment). 1 zip/tập.
os.makedirs(f"{PROJECT}/output_p1", exist_ok=True)
WANT = ["pages_han","pages_vi","split_manifest.json","vi_sentences.jsonl","vi_boxes.jsonl",
        "vi_review.jsonl","oov_vocab.jsonl"]
for VOL in VOLS:
    zp = f"{PROJECT}/output_p1/{VOL}.zip"
    if os.path.exists(zp): os.remove(zp)
    have = [f"{VOL}/{w}" for w in WANT if os.path.exists(f"{MMCY}/out/{VOL}/{w}")]
    subprocess.run(["zip","-qr",zp]+have, cwd=f"{MMCY}/out", check=True)
    print("pushed output_p1/"+VOL+".zip")
subprocess.run(["ls","-lh",f"{PROJECT}/output_p1/"])


## Xong P1
`output_p1/<vol>.zip` = đầu vào PIPELINE 2 (Hán). Mở `Minh_Menh_P2_Han.ipynb`.
